# Nurse Navigation - Behavioral Health Screen

Estimates how many 911 calls transferred to nurse navigation involve a behavioral health component. Two layers, as scoped:

- A full-population keyword pass over every nurse note, for a fast directional count.
- A large language model read on a sample that handles negation and captures behavioral health as a primary reason or as a secondary factor on a call logged under something else, with a supporting quote for every flag.

The keyword pass and the large language model read are compared on the sample to give the keyword count a measured error range rather than an unqualified estimate.

Prompts and the keyword list are stored outside the notebook and loaded at runtime, so they can be edited and version-controlled without touching the code.

All figures are directional. A flag means the note contains behavioral health language, not that the call was transferred to nurse navigation as a behavioral health case.

## 1. Setup

In [ ]:
import os, re, json, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200); pd.set_option("display.max_colwidth", 300)
plt.rcParams.update({"figure.figsize":(11,5),"figure.dpi":110,"axes.grid":True,"grid.alpha":0.25,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"axes.titlesize":13,"axes.titleweight":"bold"})
TEAL, NAVY, CORAL, GOLD, GREY = "#028090","#0B2545","#D1495B","#E0A500","#8FA0A6"

DATA_DIR   = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/data"
OUT_DIR    = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/results"
PROMPT_DIR = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/prompts"
SOURCE_FILE = "data_april2026-aug2026.xlsx"
LLM_MODEL   = "databricks-gpt-oss-120b"
SAMPLE_N    = 600
RANDOM_SEED = 42
os.makedirs(OUT_DIR, exist_ok=True)
RUN_ID = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
RESULTS = {}
def keep(d,n): RESULTS[n]=d.copy(); return d
print("run:", RUN_ID)

## 2. Load the external prompt and keyword files

The system prompt and the keyword list live in the prompts folder, not in the notebook. Editing behavioral-health definitions is done in those files. Abbreviations in the keyword list are shown with their meaning, for example od (overdose).

In [ ]:
def load_text(path):
    with open(path, "r") as f:
        return f.read()

BH_SYSTEM_PROMPT = load_text(os.path.join(PROMPT_DIR, "bh_system_prompt.txt"))
raw_kw = load_text(os.path.join(PROMPT_DIR, "bh_keywords.txt")).splitlines()
BH_KEYWORDS = [k.strip().lower() for k in raw_kw if k.strip() and not k.strip().startswith("#")]
NEGATIONS = ["denies","denied","no ","without","negative for","ruled out","not ","non-"]
KEYWORD_MEANINGS = {"od": "overdose"}
def kw_label(k): return f"{k} ({KEYWORD_MEANINGS[k]})" if k in KEYWORD_MEANINGS else k
KW_PATTERNS = {k: re.compile("(?<![a-z0-9])" + re.escape(k) + "(?:s|es)?(?![a-z0-9])") for k in BH_KEYWORDS}

print(f"system prompt: {len(BH_SYSTEM_PROMPT)} chars")
print(f"keywords loaded: {len(BH_KEYWORDS)}")
print("first 10 keywords:", [kw_label(k) for k in BH_KEYWORDS[:10]])
print("abbreviations in keyword list:", ", ".join(f"{k} = {v}" for k,v in KEYWORD_MEANINGS.items() if k in BH_KEYWORDS))

## 3. Load calls and resolve columns

In [ ]:
def clean_col(c): return re.sub(r"_+","_",re.sub(r"[^\w]+","_",str(c).strip())).lower()
raw = pd.read_excel(os.path.join(DATA_DIR, SOURCE_FILE)); raw.columns=[clean_col(c) for c in raw.columns]
def find_col(df, exact, contains=None):
    norm=lambda x:x.strip("_"); nrm={norm(c):c for c in df.columns}
    for c in exact:
        if c in df.columns: return c
        if norm(c) in nrm: return nrm[norm(c)]
    for pat in (contains or []):
        hits=[c for c in df.columns if pat in c]
        if hits: return sorted(hits,key=len)[0]
    return None
NOTES  = find_col(raw, ["nurses_notes","nurse_notes","notes"], ["nurses_note","note"])
DATE   = find_col(raw, ["transaction_create_date_time_eastern"], ["date_time"])
DISPO  = find_col(raw, ["transaction_response_names","response_macro","response"], ["response_name"])
MARKET = find_col(raw, ["market_name","market"], ["market"])
CAUSE  = find_col(raw, ["cause","chief_complaint"], ["cause","complaint"])
df = raw.copy()
if DATE: df[DATE]=pd.to_datetime(df[DATE], errors="coerce")
print(f"{len(df):,} calls")
pd.DataFrame({"field":["notes","date","disposition","market","cause"],"resolved":[NOTES,DATE,DISPO,MARKET,CAUSE]})

## 4. Check for a coded behavioral-health field

The first layer the email describes is a clean count from a coded field, if one exists. This scans the disposition and chief-complaint values for psychiatric, behavioral, or overdose categories before relying on the notes.

In [ ]:
CODED_BH_TERMS = ["psych","behavioral","mental","overdose","suicid","intox","substance"]
coded_hits = {}
for col in [DISPO, CAUSE]:
    if col:
        vals = df[col].fillna("").astype(str).str.lower()
        mask = vals.apply(lambda v: any(t in v for t in CODED_BH_TERMS))
        coded_hits[col] = int(mask.sum())
        if mask.sum():
            display(df.loc[mask, col].value_counts().head(10).rename("calls").to_frame())
print("coded behavioral-health hits by field:", coded_hits)
has_coded = any(v>0 for v in coded_hits.values())
print("A coded behavioral-health category exists." if has_coded else
      "No coded behavioral-health category found - the note-based screen below is the available measure.")

## 5. Layer 1 - full-population keyword screen

Every note is scanned for the behavioral-health terms, with negation handling so phrases like "denies suicidal ideation" are not counted. This is the fast directional count across all calls. Keywords match as whole words or whole acronyms only, with an optional plural s or es. For example, od matches "pt od" and "od'd" but not "today" or "body", and hallucination matches "hallucinations".

In [ ]:
def kw_hit(text):
    t = str(text).lower()
    for k in BH_KEYWORDS:
        for m in KW_PATTERNS[k].finditer(t):
            window = t[max(0,m.start()-15):m.start()]
            if not any(neg in window for neg in NEGATIONS):
                return True
    return False

if NOTES:
    df["bh_keyword"] = df[NOTES].fillna("").apply(kw_hit)
    kw_n = int(df["bh_keyword"].sum())
    kw_pct = kw_n/len(df)*100
    print(f"Keyword behavioral-health flags: {kw_n:,} ({kw_pct:.1f}% of all calls)")
    keep(pd.DataFrame({"metric":["total_calls","behavioral_health_keyword_flags","behavioral_health_keyword_percent"],
                       "value":[len(df), kw_n, round(kw_pct,1)]}), "bh_keyword_summary")

In [ ]:
if NOTES:
    hit_terms = {}
    lower_notes = df.loc[df["bh_keyword"], NOTES].fillna("").str.lower()
    for k in BH_KEYWORDS:
        c = int(lower_notes.str.contains(KW_PATTERNS[k].pattern, regex=True).sum())
        if c: hit_terms[k]=c
    term_df = pd.Series(hit_terms).sort_values(ascending=False).head(15).rename("calls").to_frame()
    term_df.index = [kw_label(k) for k in term_df.index]
    keep(term_df.reset_index().rename(columns={"index":"keyword"}), "bh_top_keywords")
    fig, ax = plt.subplots(figsize=(10,5)); o=term_df.sort_values("calls")
    ax.barh(range(len(o)), o["calls"], color=TEAL)
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o.index, fontsize=9)
    ax.set_title("Most frequent behavioral-health keywords (flagged notes)")
    for i,v in enumerate(o["calls"]): ax.annotate(f"{v:,}",(v,i),xytext=(4,0),textcoords="offset points",va="center",fontsize=9)
    plt.tight_layout(); plt.show()
    display(term_df)

## 6. Layer 2 - large language model read on a sample

The model reads a sample of notes using the external system prompt. It handles negation, records whether behavioral health is the primary reason or a secondary factor, assigns a category, and attaches a supporting quote. The sample is stratified so both keyword-flagged and non-flagged notes are represented, which is what allows the keyword pass to be scored for both missed and over-counted cases.

In [ ]:
from openai import OpenAI
DATABRICKS_TOKEN = (dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
                    if "dbutils" in dir() else os.environ.get("DATABRICKS_TOKEN",""))
client = OpenAI(api_key=DATABRICKS_TOKEN, base_url="https://adb-2790612761746757.17.azuredatabricks.net/serving-endpoints")
def llm_call(system, user, max_tokens=700):
    r = client.chat.completions.create(model=LLM_MODEL,
        messages=[{"role":"system","content":system},{"role":"user","content":user}], temperature=0.0, max_tokens=max_tokens)
    c=r.choices[0].message.content
    if isinstance(c,list):
        for it in c:
            if isinstance(it,dict) and it.get("type")=="text": return it.get("text","")
        return json.dumps(c)
    return c

def parse_bh(raw):
    try: o=json.loads(raw)
    except Exception: return None
    return o if "behavioral_health" in o else None

In [ ]:
usable = df[df[NOTES].fillna("").str.len()>=20].copy() if NOTES else df.copy()
flagged = usable[usable["bh_keyword"]]
not_flagged = usable[~usable["bh_keyword"]]
half = SAMPLE_N//2
parts = []
if len(flagged): parts.append(flagged.sample(min(len(flagged), half), random_state=RANDOM_SEED))
if len(not_flagged): parts.append(not_flagged.sample(min(len(not_flagged), SAMPLE_N-half), random_state=RANDOM_SEED))
sample = pd.concat(parts).reset_index(drop=True)
print(f"Sampling {len(sample):,} notes ({sample['bh_keyword'].sum()} keyword-flagged, {(~sample['bh_keyword']).sum()} not flagged)")

In [ ]:
rows=[]
for _, r in sample.iterrows():
    note = str(r.get(NOTES,""))
    try: o = parse_bh(llm_call(BH_SYSTEM_PROMPT, note))
    except Exception: o = None
    bh = (o or {}).get("behavioral_health", {})
    rows.append({
        "bh_keyword": bool(r["bh_keyword"]),
        "bh_llm": bool(bh.get("present", False)),
        "role": bh.get("role","none"),
        "category": bh.get("category","none"),
        "cause": r.get(CAUSE,""),
        "ok": o is not None,
    })
ext = pd.DataFrame(rows)
valid = ext[ext["ok"]].copy()
print(f"Valid large language model reads: {ext['ok'].mean()*100:.1f}%  ({len(valid)} of {len(ext)})")

## 7. Score the keyword pass against the large language model read

On the sample, the large language model read is treated as the reference. This produces the error range the email asks for: how often the keyword pass agrees, how many behavioral-health notes it misses, and how many it over-counts.

In [ ]:
if len(valid):
    tp = int(((valid["bh_keyword"]) & (valid["bh_llm"])).sum())
    fp = int(((valid["bh_keyword"]) & (~valid["bh_llm"])).sum())
    fn = int(((~valid["bh_keyword"]) & (valid["bh_llm"])).sum())
    tn = int(((~valid["bh_keyword"]) & (~valid["bh_llm"])).sum())
    precision = tp/(tp+fp)*100 if (tp+fp) else 0
    recall    = tp/(tp+fn)*100 if (tp+fn) else 0
    agree     = (tp+tn)/len(valid)*100
    score = pd.DataFrame({"metric":["agreement_percent","keyword_precision_percent","keyword_recall_percent",
                                    "large_language_model_behavioral_health_rate_in_sample_percent"],
                          "value":[round(agree,1), round(precision,1), round(recall,1),
                                   round(valid["bh_llm"].mean()*100,1)]})
    keep(score, "bh_keyword_vs_llm")
    conf = pd.DataFrame([[tp,fp],[fn,tn]],
        index=["keyword: yes","keyword: no"], columns=["large language model: yes","large language model: no"])
    keep(conf.reset_index().rename(columns={"index":"keyword"}), "bh_confusion")
    display(conf); display(score)
    print(f"Keyword precision {precision:.0f}% (share of keyword flags the large language model confirms).")
    print(f"Keyword recall {recall:.0f}% (share of large language model behavioral-health notes the keyword pass caught).")

## 8. Adjusted full-population estimate

The keyword count is adjusted by the precision measured on the sample, giving a directional behavioral-health volume with a stated basis rather than a raw keyword number.

In [ ]:
if len(valid) and NOTES:
    prec = precision/100 if (tp+fp) else 0
    llm_rate = valid["bh_llm"].mean()
    adj_from_kw = int(kw_n * prec)
    adj_from_rate = int(len(df) * llm_rate)
    est = pd.DataFrame({
        "estimate":["Keyword flags (raw)","Keyword flags adjusted by precision","Large language model sample rate applied to all calls"],
        "calls":[kw_n, adj_from_kw, adj_from_rate],
        "percent_of_all_calls":[round(kw_n/len(df)*100,1), round(adj_from_kw/len(df)*100,1), round(adj_from_rate/len(df)*100,1)],
    })
    keep(est, "bh_population_estimate")
    fig, ax = plt.subplots(figsize=(9,3.6))
    ax.barh(range(len(est)), est["calls"], color=[GREY,TEAL,NAVY])
    ax.set_yticks(range(len(est))); ax.set_yticklabels(est["estimate"], fontsize=10)
    ax.set_title("Behavioral-health volume - three directional estimates")
    for i,v in enumerate(est["calls"]): ax.annotate(f"{v:,}",(v,i),xytext=(4,0),textcoords="offset points",va="center",fontsize=9)
    plt.tight_layout(); plt.show()
    display(est)

## 9. Primary vs secondary, category, and trend

In [ ]:
if len(valid):
    bh_only = valid[valid["bh_llm"]]
    if len(bh_only):
        role = (bh_only["role"].value_counts().rename("calls").to_frame()
                .assign(**{"% of behavioral health sample":lambda d:(d["calls"]/len(bh_only)*100).round(1)}))
        keep(role.reset_index().rename(columns={"index":"role"}), "bh_role_split")
        cat = (bh_only["category"].value_counts().rename("calls").to_frame()
               .assign(**{"% of behavioral health sample":lambda d:(d["calls"]/len(bh_only)*100).round(1)}))
        keep(cat.reset_index().rename(columns={"index":"category"}), "bh_category_split")
        display(role); display(cat)
        print("Primary means behavioral health was the reason for the call; secondary means it appeared on a call logged under something else.")

In [ ]:
if NOTES and DATE:
    m = df.dropna(subset=[DATE]).copy()
    m["month"] = m[DATE].dt.to_period("M").astype(str)
    trend = m.groupby("month")["bh_keyword"].agg(["sum","count"])
    trend["bh_pct"] = (trend["sum"]/trend["count"]*100).round(1)
    keep(trend.reset_index().rename(columns={"sum":"behavioral_health_keyword_calls","count":"total_calls",
                                             "bh_pct":"behavioral_health_keyword_percent"}), "bh_trend")
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(range(len(trend)), trend["bh_pct"], marker="o", lw=2, color=TEAL)
    ax.set_xticks(range(len(trend))); ax.set_xticklabels(trend.index, rotation=60, fontsize=8)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title("Behavioral-health keyword rate by month"); ax.set_ylabel("% of calls")
    plt.tight_layout(); plt.show()

## 10. Evidence sample

A sample of flagged notes with the model's category and supporting quote, so any behavioral-health count can be traced to the note text.

In [ ]:
ev_rows=[]
for _, r in sample.head(40).iterrows():
    note = str(r.get(NOTES,""))
    try: o = parse_bh(llm_call(BH_SYSTEM_PROMPT, note))
    except Exception: o = None
    if o and o.get("behavioral_health",{}).get("present"):
        q = (o.get("evidence") or [{}])
        ev_rows.append({"role":o["behavioral_health"].get("role"),
                        "category":o["behavioral_health"].get("category"),
                        "quote": (q[0].get("quote","") if q else "")[:160],
                        "note": note[:160]})
evidence = pd.DataFrame(ev_rows)
keep(evidence, "bh_evidence_sample")
display(evidence.head(20))

## 11. Write the workbook

In [ ]:
def sanitize(d):
    o=d.copy(); o.columns=[str(c) for c in o.columns]
    for c in o.columns:
        if o[c].dtype==object: o[c]=o[c].apply(lambda x:"" if x is None or (isinstance(x,float) and pd.isna(x)) else str(x))
    return o
TABS=[("bh_keyword_summary","Behavioral Health Summary"),("bh_top_keywords","Behavioral Health Top Keywords"),
      ("bh_keyword_vs_llm","Keyword vs Large Language Model"),("bh_confusion","Confusion"),
      ("bh_population_estimate","Population Estimate"),("bh_role_split","Primary vs Secondary"),
      ("bh_category_split","Behavioral Health Categories"),("bh_trend","Behavioral Health Trend"),("bh_evidence_sample","Evidence Sample")]
xlsx=os.path.join(OUT_DIR, f"Nurse_Navigation_Behavioral_Health_{RUN_ID}.xlsx")
try: import xlsxwriter; eng="xlsxwriter"
except ImportError: eng="openpyxl"
with pd.ExcelWriter(xlsx, engine=eng) as w:
    pd.DataFrame({"Nurse Navigation - Behavioral Health Screen":[f"Run {RUN_ID}", f"Calls: {len(df):,}",
        "Two layers: full-population keyword pass and large language model sample read.",
        "Directional. A flag means the note contains behavioral-health language, not a confirmed behavioral health transfer.",
        "Abbreviations: od = overdose.",
        "Keywords match as whole words or whole acronyms only, with an optional plural s or es.",
        "Prompts loaded from the external prompts folder."]}).to_excel(w, sheet_name="Start Here", index=False)
    for stem,tab in TABS:
        d=RESULTS.get(stem)
        if d is not None and len(d): sanitize(d).to_excel(w, sheet_name=tab[:31], index=False); print("added",tab)
for old in glob.glob(os.path.join(OUT_DIR,"Nurse_Nav_Behavioral_Health_*.xlsx"))+glob.glob(os.path.join(OUT_DIR,"Nurse_Navigation_Behavioral_Health_*.xlsx")):
    if os.path.abspath(old)!=os.path.abspath(xlsx):
        try: os.remove(old)
        except Exception: pass
print("workbook:", os.path.basename(xlsx))